In [31]:
##CELL FOR IMPORTS AND GLOBAL VARIABLES

import unittest
from collections.abc import Iterable, Sized, Sequence, Mapping
from cerberus import errors, validator

_str_type = str

sample_schema = {
    'a_string': {'type': 'string', 'minlength': 2, 'maxlength': 10},
    'a_binary': {'type': 'binary', 'minlength': 2, 'maxlength': 10},
    'a_nullable_integer': {'type': 'integer', 'nullable': True},
    'an_integer': {'type': 'integer', 'min': 1, 'max': 100},
    'a_restricted_integer': {'type': 'integer', 'allowed': [-1, 0, 1]},
    'a_boolean': {'type': 'boolean', 'meta': 'can haz two distinct states'},
    'a_datetime': {'type': 'datetime', 'meta': {'format': '%a, %d. %b %Y'}},
    'a_float': {'type': 'float', 'min': 1, 'max': 100},
    'a_number': {'type': 'number', 'min': 1, 'max': 100},
    'a_set': {'type': 'set'},
    'one_or_more_strings': {'type': ['string', 'list'], 'schema': {'type': 'string'}},
    'a_regex_email': {
        'type': 'string',
        'regex': r'^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$',
    },
    'a_readonly_string': {'type': 'string', 'readonly': True},
    'a_restricted_string': {'type': 'string', 'allowed': ['agent', 'client', 'vendor']},
    'an_array': {'type': 'list', 'allowed': ['agent', 'client', 'vendor']},
    'an_array_from_set': {
        'type': 'list',
        'allowed': set(['agent', 'client', 'vendor']),
    },
    'a_list_of_dicts': {
        'type': 'list',
        'schema': {
            'type': 'dict',
            'schema': {
                'sku': {'type': 'string'},
                'price': {'type': 'integer', 'required': True},
            },
        },
    },
    'a_list_of_values': {
        'type': 'list',
        'items': [{'type': 'string'}, {'type': 'integer'}],
        'allowed': [(),(3,4)],
        'contains':[1]
    },
    'a_list_of_integers': {'type': 'list', 'schema': {'type': 'integer'}},
    'a_dict': {
        'type': 'dict',
        'schema': {
            'address': {'type': 'string'},
            'city': {'type': 'string', 'required': True},
        },
    },
    'a_dict_with_valuesrules': {'type': 'dict', 'valuesrules': {'type': 'integer'}},
    'a_list_length': {
        'type': 'list',
        'schema': {'type': 'integer'},
        'minlength': 2,
        'maxlength': 5,
    },
    'a_nullable_field_without_type': {'nullable': True},
    'a_not_nullable_field_without_type': {},
}

val = validator.Validator(sample_schema)
val.document = {}
tcase = unittest.TestCase()

MUMCUT:

1. Test for LIF: Given a minimal DNF representation of a predicate f , for each implicant i, choose unique true points (UTPs) such that Literals not in i take on values T and F.


2. Test for LOF: Given a minimal DNF representation of a predicate f , for each literal c in each implicant i, TR contains a unique true point for i and a near false point for c in i such that the two points differ only in the truth value of c.


3. Test for infeasible cases: Given a minimal DNF representation of a predicate f , for each literal c in each implicant i, choose near false points (NFPs) such that clauses not in i take on values T and F.



**_validate_allowed**

*Requirements:*

1. If the given value is an iterable other than string through all of it's elements, if any of them are not within "allowed_values" raises an UNALLOWED_VALUES error,
2. If the given value is not an iterable or if it's a string, checks if the value itself is within "allowed_values" and if it isn't raises an UNALLOWED_VALUES error,
3. If none of the conditions are met doesnt raise an error or give an output.

*Logic Equations:*
- $ A = $ "value" is an iterable other than str
- $ B = $ "value" is an element of "allowed_values"
- $ C = $ "value" is a subset of "allowed values"
- $ F = $ "_validate_allowed" method does NOT raise an UNALLOWED_VALUES error with the given parameters

$ F = A C +  \bar{A} B$

*Test Cases:*

- MUTP:
    - Implicant $ = A C \longrightarrow AC = 11, B = 0/1 $
        1. $ABC = 110$ 
        2. $ABC = 111$ 
    - Implicant $ =  \bar{A} B \longrightarrow AB = 01, C = 0/1 $
        1. $ABC = 010$ 
        2. $ABC = 011$
- CUTPNFP:
    - Literal $ = A$ and Implicant $ = A C,$ True Points: $AC = 11, B = 0/1,$ Near false points:
        1. $ABC = 010$
    - Literal $ = A$ and Implicant $ = \bar{A} B ,$ True Points: $AB = 01, C = 0/1,$ Near false points:
        1. $ABC = 110$
    - Literal $ = B$ and Implicant $ = \bar{A} B,$ True Points: $AB = 01, C = 0/1,$ Near false points:
        1. $ABC = 000$ 
        2. $ABC = 001$ 
    - Literal $ = C$ and Implicant $ = AC,$ True Points: $AC = 11, B = 0/1,$ Near false points:
        1. $ABC = 100$ 
        2. $ABC = 110$
- MNFP:
    - Literal $ = A$ and Implicant $ = A C,$ True Points: $AC = 11, Clause = B,$ Near false points:
        1. $ABC = 010$
        2. $ABC = 000$
    - Literal $ = A$ and Implicant $ = \bar{A} B ,$ True Points: $AB = 01, Clause = C,$ Near false points:
        1. $ABC = 110$
        2. $ABC = 111$
    - Literal $ = B$ and Implicant $ = \bar{A} B,$ True Points: $AB = 01, C = 0/1,$ Near false points:
        1. $ABC = 000$ 
        2. $ABC = 001$ 
    - Literal $ = C$ and Implicant $ = AC,$ True Points: $AC = 11, B = 0/1,$ Near false points:
        1. $ABC = 100$ 
        2. $ABC = 110$

This should be all the required test cases for MUMCUT, but it should be noted that there is severe overlap between these.

The unique cases within these are found in the code cell below: 

In [32]:
newlis = list()
alllis = "110,111,010,011,010,110,000,001,100,110,010,000,110,111,000,001,100,110"
alllis = alllis.split(",")
for ele in alllis:
    if ele not in newlis:
        newlis.append(ele)
print(newlis)
print(len(newlis))

['110', '111', '010', '011', '000', '001', '100']
7


In [33]:
# def _validate_allowed(self, allowed_values, field, value):
#         """{'type': 'container'}"""
#         if isinstance(value, Iterable) and not isinstance(value, _str_type):
#             unallowed = tuple(x for x in value if x not in allowed_values)
#             if unallowed:
#                 self._error(field, errors.UNALLOWED_VALUES, unallowed)
#         else:
#             if value not in allowed_values:
#                 self._error(field, errors.UNALLOWED_VALUE, value)

In [34]:
# A = "value" is an iterable other than str
# B = "value" is an element of "allowed_values"
# C = "value" is a subset of "allowed values"
# F = "_validate_allowed" method does NOT raise an UNALLOWED_VALUES error with the given parameters

# F = AC + B!A

allowed_values = []
field = 'a_list_of_values'
value = None
for ele in newlis:
    if ele[0] == "0":
        value = "beans"
    else: value = ['al1', 'al2']
    if ele[1] == "1":
        allowed_values = [1, 2, value]
    else:
        allowed_values = [1, 2]
    if ele[2] == "1":
        for elem in value:
            allowed_values.append(elem)
    tcase.assertLogs(val._validate_allowed(allowed_values,field,value))


**_validate_max**

*Requirements:*

1. If the given value is greater than the given max_value raises a MAX_VALUE error,
2. If the given value and the max_value are NOT comparable types does NOT raise a TypeError,
3. If neither condition is met doesnt raise an error or give an output.

*Logic Equations:*
- $ A = $ "value" is greater than "max_value"
- $ B = $ "value" and "max_value" are not comparable types
- $ F = $ "_validate_max" method does NOT raise an MAX_VALUE error with the given parameters

$ F = \bar{A}$

The function uses a try/except method to supress TypeErrors so B is not relevant to the final equation. Weird choice by the developer but it works.

*Test Cases:*

- MUTP:
    - Implicant $ = \bar{A} \longrightarrow A = 0 $
        1. $AB = 01$ 
        2. $AB = 00$ 
- CUTPNFP:
    - Literal $ = A$ and Implicant $ = \bar{A},$ True Points: $A = 0, B = 0/1,$ Near false points:
        1. $AB = 11$ 
        2. $AB = 10$
- MNFP:
    - Literal $ = A$ and Implicant $ = \bar{A},$ True Points: $A = 0, B = 0/1,$ Near false points:
        1. $AB = 11$ 
        2. $AB = 10$ 

This function doesn't require many test cases as the equation is expected to depend on only one of the requirements.

Unique cases: 01, 00, 11, 10

In [35]:
# def _validate_max(self, max_value, field, value):
#         """{'nullable': False }"""
#         try:
#             if value > max_value:
#                 self._error(field, errors.MAX_VALUE)
#         except TypeError:
#             pass

In [36]:
# A = "value" is greater than "max_value"
# B = "value" and "max_value" are not comparable types
# F = "_validate_max" method does NOT raise an MAX_VALUE error with the given parameters

cases = "01,00,11,10"
splitcases = cases.split(",")

field = 'a_number'
value = None
max_value = 2
for case in splitcases:
    if case[0] == "0":
        value = 1
    else:
        value = 5
    if case[1] == "1":
        value = f"{value}"
    
    tcase.assertLogs(val._validate_max(max_value,field,value))

**_validate_contains**

Requirements:
1. If value is not an iterable returns None,
2. If expected_values is an iterable (excluding str) converts expected_values into an unordered list consisting of it's elements, otherwise converts into an unordered list with it's inital value as the only element. Then compares the resulting unordered list to an unordered list of elements within value to determine if expected_values is a subset of value. If expected_values is NOT a subset of value raises a MISSING_MEMBERS error,
3. If neither condition is met doesn't raise an error or give an output.

*Logic Equations*
- $ A = $ "value" is an iterable
- $ B = $ "expected_value" is a non string iterable 
- $ C = $ set created from "expected_values" is a subset of "value"
- $ F = $ "_validate_contains" method does NOT raise an MISSING_MEMBERS error with the given parameters

$ F = AC$

The function uses B only when creating a set from "expected_values", thus it doesnt factor into final equation.

*Test Cases:*

- MUTP:
    - Implicant $ = AC \longrightarrow A = 1, C = 1 $
        1. $ABC = 101$ 
        2. $ABC = 111$ 
- CUTPNFP:
    - Literal $ = A$ and Implicant $ = AC,$ True Points: $AC = 11, B = 0/1,$ Near false points:
        1. $ABC = 001$ 
        2. $ABC = 011$
    - Literal $ = C$ and Implicant $ = AC,$ True Points: $AC = 11, B = 0/1,$ Near false points:
        1. $ABC = 100$ 
        2. $ABC = 110$
- MNFP:
    - Literal $ = A$ and Implicant $ = AC,$ True Points: $AC = 11, B = 0/1,$ Near false points:
        1. $ABC = 001$ 
        2. $ABC = 011$ 
    - Literal $ = C$ and Implicant $ = AC,$ True Points: $AC = 11, B = 0/1,$ Near false points:
        1. $ABC = 100$ 
        2. $ABC = 110$ 



In [37]:
newlis = list()
alllis = "101,111,001,011,100,110,001,011,100,110"
alllis = alllis.split(",")
for ele in alllis:
    if ele not in newlis:
        newlis.append(ele)
print(newlis)
print(len(newlis))

['101', '111', '001', '011', '100', '110']
6


In [38]:
# def _validate_contains(self, expected_values, field, value):
#         """{'empty': False }"""
#         if not isinstance(value, Iterable):
#             return

#         if not isinstance(expected_values, Iterable) or isinstance(
#             expected_values, _str_type
#         ):
#             expected_values = set((expected_values,))
#         else:
#             expected_values = set(expected_values)

#         missing_values = expected_values - set(value)
#         if missing_values:
#             self._error(field, errors.MISSING_MEMBERS, missing_values)


In [39]:
# A = "value" is an iterable
# B = "expected_value" is a non string iterable 
# C = set created from "expected_values" is a subset of "value"
# F = "_validate_contains" method does NOT raise an MISSING_MEMBERS error with the given parameters

expected_values = None
field = 'a_list_of_values'
value = None
for case in newlis:
    if case[0] == "0":
        value = 1
    else:
        value = [1,2,3]
    if case[1] == "1":
        if case[2] == "1":
            expected_values = [1,2]
        else:
            expected_values = [4,5]
    else:
        if case[2] == "1":
            expected_values = 3
        else:
            expected_values = "asb"
    tcase.assertLogs(val._validate_contains(expected_values, field, value))

**_validate_empty**

Requirements:
1. If given "value" parameter provides a len() method and the len() method returns 0 removes the validation checks for the "allowed, forbidden, items, minlength, maxlength, regex, check_with" rules from the queue, then if the given "empty" parameter is False raises an EMPTY_NOT_ALLOWED error.
2. If both conditions aren't met at the same time doesn't raise an error or provide an output.

*Logic Equations*
- $ A = $ "value" provides a len() method & has a length of 0
- $ B = $ "empty" is True
- $ F = $ "_validate_empty" method does NOT raise an EMPTY_NOT_ALLOWED error with the given parameters

$ F = \bar{A} + B$

*Test Cases:*

- MUTP:
    - Implicant $ = \bar{A} + B\longrightarrow A = 0, B = 1 $
        1. $AB = 01$ 
- CUTPNFP:
    - Literal $ = A$ and Implicant $ = \bar{A} + B,$ True Points: $AB = 01,$ Near false points:
        1. $AB = 11$
    - Literal $ = B$ and Implicant $ = \bar{A} + B,$ True Points: $AB = 01,$ Near false points:
        1. $AB = 00$
- MNFP:
    - Literal $ = A$ and Implicant $ = \bar{A} + B,$ True Points: $AB = 01,$ Near false points:
        1. $AB = 11$
    - Literal $ = B$ and Implicant $ = \bar{A} + B,$ True Points: $AB = 01,$ Near false points:
        1. $AB = 00$

Not many test cases since this equation does depend on all requirements.

In [40]:
# def _validate_empty(self, empty, field, value):
#         """{'type': 'boolean'}"""
#         if isinstance(value, Sized) and len(value) == 0:
#             self._drop_remaining_rules(
#                 'allowed',
#                 'forbidden',
#                 'items',
#                 'minlength',
#                 'maxlength',
#                 'regex',
#                 'check_with',
#             )
#             if not empty:
#                 self._error(field, errors.EMPTY_NOT_ALLOWED)

# # Sized is an abstract class method used to check if an abc provides the len() method

# # def _drop_remaining_rules(self, *rules):
# #         """
# #         Drops rules from the queue of the rules that still need to be evaluated for the
# #         currently processed field. If no arguments are given, the whole queue is
# #         emptied.
# #         """
# #         if rules:
# #             for rule in rules:
# #                 try:
# #                     self._remaining_rules.remove(rule)
# #                 except ValueError:
# #                     pass
# #         else:
# #             self._remaining_rules = []


In [41]:
# A = "value" provides a len() method & has a length of 0
# B = "empty" is True
# F = "_validate_empty" method does NOT raise an EMPTY_NOT_ALLOWED error with the given parameters

cases = "01,00,11"
splitcases = cases.split(",")

field = 'a_list_of_values'
value = None
empty = None
for case in splitcases:
    if case[0] == "0":
        value = 8
    else:
        value = []
    if case[1] == "1":
        empty = True
    else:
        empty = False
    
    tcase.assertLogs(val._validate_empty(empty, field, value))

**_validate_schema**

Requirements:
1. If the given "schema" is None then returns None,
2. If the given "value" parameter is an ordered collection of elements (collections.Sequence) excluding a str calls the __validate_schema_sequence method with it's own input parameters,
3. Otherwise if the given "value" parameter is a collection of elements that provides key lookups (collections.Mapping) calls the __validate_schema_mapping method with it's own input parameters.
4. Does not raise an error or provide an output value.

*Logic Equations*
- $ A = $ "schema" is None
- $ B = $ "value" is a Sequence other than string
- $ C = $ "value" provides key lookups
- $ F = $ "_validate_schema" method calls either "__validate_schema_sequence" or "__validate_schema_mapping"

$ F = \bar{A}(B + C)$


*Test Cases:*

- MUTP:
    - Implicant $ = \bar{A}(B + C)\longrightarrow A = 0, BC = 01/10/11 $
        1. $ABC = 011$ 
- CUTPNFP:
    - Literal $ = A$ and Implicant $ = \bar{A}(B + C),$ True Points: $ABC = 010/011/001,$ Near false points:
        1. $ABC = 110$
        2. $ABC = 111$
        3. $ABC = 101$
    - Literal $ = B$ and Implicant $ = \bar{A}(B + C),$ True Points: $ABC = 010/011/001,$ Near false points:
        1. $ABC = 000$
        2. $ABC = 101$
        3. $ABC = 100$
    - Literal $ = C$ and Implicant $ = \bar{A}(B + C),$ True Points: $ABC = 010/011/001,$ Near false points:
        1. $ABC = 000$
        2. $ABC = 110$
        3. $ABC = 100$
- MNFP:
    - Literal $ = A$ and Implicant $ = \bar{A}(B + C),$ True Points: $ABC = 010/011/001,$ Near false points:
        1. $ABC = 110$
        2. $ABC = 101$
    - Literal $ = B$ and Implicant $ = \bar{A}(B + C),$ True Points: $ABC = 010/011/001,$ Near false points:
        1. $ABC = 000$
        3. $ABC = 100$
    - Literal $ = C$ and Implicant $ = \bar{A}(B + C),$ True Points: $ABC = 010/011/001,$ Near false points:
        1. $ABC = 010$
        3. $ABC = 100$

There is overlap between these cases, so:

In [42]:
newlis = list()
alllis = "011,110,111,101,000,101,100,000,110,100,110,101,000,100,010,100"
alllis = alllis.split(",")
for ele in alllis:
    if ele not in newlis:
        newlis.append(ele)
print(newlis)
print(len(newlis))

['011', '110', '111', '101', '000', '100', '010']
7


In [43]:
# def _validate_schema(self, schema, field, value):
#         """
#         {'type': ['dict', 'string'],
#          'anyof': [{'check_with': 'schema'},
#                    {'check_with': 'bulk_schema'}]}
#         """
#         if schema is None:
#             return

#         if isinstance(value, Sequence) and not isinstance(value, _str_type):
#             self.__validate_schema_sequence(field, schema, value)
#         elif isinstance(value, Mapping):
#             self.__validate_schema_mapping(field, schema, value)


In [44]:
# A = "schema" is None
# B = "value" is a Sequence other than string
# C = "value" provides key lookups
# F = "_validate_schema" method calls either "__validate_schema_sequence" or "__validate_schema_mapping"


schema = None
field = 'a_list_of_values'
value = None
for case in newlis:
    if case[0] == "0":
        schema = sample_schema
    else:
        schema = None
    if case[1] == "1":
        if case[2] == "1":
            value = dict()
        else:
            value = []
    else:
        value = "123"
    tcase.assertLogs(val._validate_schema(schema, field, value))